# Indoor Localization - Exploratory Data Analysis

This notebook provides exploratory data analysis for the UJIIndoorLoc dataset and demonstrates the indoor localization pipeline.

In [3]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Add project root to path
import sys
sys.path.append('..')

# Import our modules
from src.data_io import load_uji
from src.utils_geo import add_xy
from src.features import RssiCleaner, APSelector, FeatureEngineer
from src.baselines import knn_building_pipeline, knn_floor_pipeline, knn_xy_pipeline
from src.evaluate import evaluate_split
from src.metrics import print_evaluation_report

## 1. Load and Explore Data

In [4]:
# Load the data
data_dir = Path('../data')
train, val = load_uji(str(data_dir / 'TrainingData.csv'), 
                      str(data_dir / 'ValidationData.csv'))

print(f"Training set: {train.shape[0]} samples, {train.shape[1]} features")
print(f"Validation set: {val.shape[0]} samples, {val.shape[1]} features")
print(f"\nColumns: {train.columns.tolist()[:10]}...")

Training set: 19937 samples, 529 features
Validation set: 1111 samples, 529 features

Columns: ['WAP001', 'WAP002', 'WAP003', 'WAP004', 'WAP005', 'WAP006', 'WAP007', 'WAP008', 'WAP009', 'WAP010']...


In [ ]:
# Basic statistics
print("Building distribution:")
print(train['BUILDINGID'].value_counts().sort_index())
print(f"\nFloor distribution:")
print(train['FLOOR'].value_counts().sort_index())
print(f"\nBuildings and floors:")
print(train.groupby('BUILDINGID')['FLOOR'].value_counts().sort_index())

In [5]:
# Coordinate conversion
train_xy, center = add_xy(train)
val_xy, _ = add_xy(val)

print(f"Reference point: lat={center[0]:.2f}, lon={center[1]:.2f}")
print("Coordinate ranges:")
print(train_xy[['X_M', 'Y_M']].describe())

Reference point: lat=4864852.15, lon=-7423.06
Coordinate ranges:
                X_M           Y_M
count  19937.000000  19937.000000
mean     -41.215047     18.738987
std      123.402010     66.933183
min     -268.277500   -106.404684
25%     -171.676100    -31.274400
50%        0.000000      0.000000
75%       63.867900     77.740600
max      122.241910    164.538100


## 2. Wi-Fi Signal Analysis

In [ ]:
# Analyze RSSI distributions
wap_cols = [c for c in train.columns if c.startswith('WAP')]
rssi_data = train[wap_cols].replace(-100, np.nan)  # Missing values

plt.figure(figsize=(15, 10))

# RSSI distribution
plt.subplot(2, 2, 1)
rssi_flat = rssi_data.values.flatten()
rssi_flat = rssi_flat[~np.isnan(rssi_flat)]
plt.hist(rssi_flat, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('RSSI (dBm)')
plt.ylabel('Frequency')
plt.title('RSSI Distribution (All APs)')
plt.grid(True, alpha=0.3)

# AP coverage
plt.subplot(2, 2, 2)
coverage = (train[wap_cols] > -110).mean().sort_values(ascending=False)
plt.hist(coverage.values, bins=30, alpha=0.7, edgecolor='black')
plt.xlabel('Coverage (fraction of samples)')
plt.ylabel('Number of APs')
plt.title('AP Coverage Distribution')
plt.grid(True, alpha=0.3)

# Top 20 APs by coverage
plt.subplot(2, 2, 3)
top_aps = coverage.head(20)
plt.bar(range(len(top_aps)), top_aps.values)
plt.xticks(range(len(top_aps)), top_aps.index, rotation=45, ha='right')
plt.ylabel('Coverage')
plt.title('Top 20 APs by Coverage')
plt.grid(True, alpha=0.3)

# RSSI by building
plt.subplot(2, 2, 4)
for building in sorted(train['BUILDINGID'].unique()):
    building_data = train[train['BUILDINGID'] == building]
    rssi_building = building_data[wap_cols].replace(-100, np.nan).values.flatten()
    rssi_building = rssi_building[~np.isnan(rssi_building)]
    if len(rssi_building) > 0:
        plt.hist(rssi_building, bins=30, alpha=0.5, label=f'Building {building}',
                density=True)
plt.xlabel('RSSI (dBm)')
plt.ylabel('Density')
plt.title('RSSI Distribution by Building')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Spatial Analysis

In [ ]:
# Plot spatial distribution
plt.figure(figsize=(15, 10))

# All points
plt.subplot(2, 2, 1)
plt.scatter(train_xy['X_M'], train_xy['Y_M'], alpha=0.1, s=1)
plt.xlabel('X (meters)')
plt.ylabel('Y (meters)')
plt.title('All Training Points')
plt.axis('equal')
plt.grid(True, alpha=0.3)

# By building
plt.subplot(2, 2, 2)
for building in sorted(train_xy['BUILDINGID'].unique()):
    mask = train_xy['BUILDINGID'] == building
    plt.scatter(train_xy.loc[mask, 'X_M'], train_xy.loc[mask, 'Y_M'], 
               alpha=0.3, s=2, label=f'Building {building}')
plt.xlabel('X (meters)')
plt.ylabel('Y (meters)')
plt.title('Points by Building')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)

# By floor (first building only)
plt.subplot(2, 2, 3)
building_0 = train_xy[train_xy['BUILDINGID'] == 0]
for floor in sorted(building_0['FLOOR'].unique()):
    mask = building_0['FLOOR'] == floor
    plt.scatter(building_0.loc[mask, 'X_M'], building_0.loc[mask, 'Y_M'],
               alpha=0.5, s=3, label=f'Floor {floor}')
plt.xlabel('X (meters)')
plt.ylabel('Y (meters)')
plt.title('Building 0 - Points by Floor')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)

# Point density
plt.subplot(2, 2, 4)
plt.hist2d(train_xy['X_M'], train_xy['Y_M'], bins=50, cmap='viridis')
plt.colorbar(label='Point density')
plt.xlabel('X (meters)')
plt.ylabel('Y (meters)')
plt.title('Point Density Heatmap')
plt.axis('equal')

plt.tight_layout()
plt.show()

## 4. Feature Engineering Demonstration

In [ ]:
# Demonstrate preprocessing pipeline
print("Original data shape:", train.shape)

# RSSI cleaning
cleaner = RssiCleaner()
train_clean = cleaner.fit_transform(train)
print("After RSSI cleaning:", train_clean.shape)

# AP selection
selector = APSelector(coverage_min=0.02, top_k=100)
train_selected = selector.fit_transform(train_clean)
print(f"After AP selection ({len(selector.keep_cols)} APs):", train_selected.shape)

# Feature engineering
engineer = FeatureEngineer()
train_features = engineer.fit_transform(train_selected)
print("After feature engineering:", train_features.shape)

# Show new features
print("\nNew features:")
print(train_features[['ap_count', 'rssi_mean', 'rssi_median', 'rssi_max']].describe())

## 5. Model Training and Evaluation

In [ ]:
# Train and evaluate baseline models
wap_cols = [c for c in train.columns if c.startswith('WAP')]
X_tr = train[wap_cols]
X_va = val[wap_cols]

# Use smaller sample for faster training in notebook
sample_size = 5000
if len(X_tr) > sample_size:
    sample_idx = np.random.choice(len(X_tr), sample_size, replace=False)
    X_tr = X_tr.iloc[sample_idx]
    y_b_tr = train.iloc[sample_idx]['BUILDINGID']
    y_f_tr = train.iloc[sample_idx]['FLOOR']
    xy_tr = train_xy.iloc[sample_idx][['X_M', 'Y_M']].values
else:
    y_b_tr = train['BUILDINGID']
    y_f_tr = train['FLOOR']
    xy_tr = train_xy[['X_M', 'Y_M']].values

y_b_va = val['BUILDINGID']
y_f_va = val['FLOOR']
xy_va = val_xy[['X_M', 'Y_M']].values

print(f"Training on {len(X_tr)} samples, validating on {len(X_va)} samples")

In [ ]:
# Train models
print("Training building classifier...")
building_model = knn_building_pipeline().fit(X_tr, y_b_tr)

print("Training floor classifier...")
floor_model = knn_floor_pipeline().fit(X_tr, y_f_tr)

print("Training position regressor...")
xy_model = knn_xy_pipeline().fit(X_tr, xy_tr)

print("Training completed!")

In [ ]:
# Evaluate models
print("Evaluating models...")
metrics = evaluate_split(building_model, floor_model, xy_model,
                        X_va, y_b_va, y_f_va, xy_va)

print("\nEvaluation Results:")
for k, v in metrics.items():
    if 'pct' in k:
        print(f"  {k}: {v:.1f}")
    else:
        print(f"  {k}: {v:.3f}")

# Detailed report
print("\nDetailed Report:")
b_pred = building_model.predict(X_va)
f_pred = floor_model.predict(X_va)
xy_pred = xy_model.predict(X_va)
print_evaluation_report(y_b_va, b_pred, y_f_va, f_pred, xy_va, xy_pred)

## 6. Error Analysis

In [ ]:
# Analyze prediction errors
from utils_geo import meter_error

errors = meter_error(xy_va, xy_pred)
building_errors = pd.DataFrame({
    'error': errors,
    'building_true': y_b_va,
    'building_pred': b_pred,
    'floor_true': y_f_va,
    'floor_pred': f_pred,
    'correct_building': y_b_va == b_pred,
    'correct_floor': y_f_va == f_pred,
    'correct_both': (y_b_va == b_pred) & (y_f_va == f_pred)
})

plt.figure(figsize=(15, 10))

# Error distribution
plt.subplot(2, 3, 1)
plt.hist(errors, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Position Error (meters)')
plt.ylabel('Frequency')
plt.title('Position Error Distribution')
plt.grid(True, alpha=0.3)

# Error by building
plt.subplot(2, 3, 2)
building_errors.boxplot(column='error', by='building_true')
plt.xlabel('Building')
plt.ylabel('Position Error (meters)')
plt.title('Error by Building')
plt.suptitle('')
plt.grid(True, alpha=0.3)

# Error vs correct building/floor
plt.subplot(2, 3, 3)
building_errors.boxplot(column='error', by='correct_both')
plt.xlabel('Correct Building+Floor')
plt.ylabel('Position Error (meters)')
plt.title('Error vs Classification Accuracy')
plt.suptitle('')
plt.grid(True, alpha=0.3)

# Confusion matrices
from sklearn.metrics import confusion_matrix

plt.subplot(2, 3, 4)
cm_building = confusion_matrix(y_b_va, b_pred)
sns.heatmap(cm_building, annot=True, fmt='d', cmap='Blues',
           xticklabels=sorted(y_b_va.unique()),
           yticklabels=sorted(y_b_va.unique()))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Building Confusion Matrix')

plt.subplot(2, 3, 5)
cm_floor = confusion_matrix(y_f_va, f_pred)
sns.heatmap(cm_floor, annot=True, fmt='d', cmap='Greens',
           xticklabels=sorted(y_f_va.unique()),
           yticklabels=sorted(y_f_va.unique()))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Floor Confusion Matrix')

# Error statistics
plt.subplot(2, 3, 6)
error_stats = building_errors.groupby('correct_both')['error'].agg(
    ['mean', 'median', 'std', 'count'])
error_stats.plot(kind='bar', ax=plt.gca())
plt.xlabel('Correct Building+Floor')
plt.ylabel('Position Error (meters)')
plt.title('Error Statistics')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Error Statistics:")
print(error_stats)

## 7. Summary

This notebook demonstrated:
1. **Data Loading**: UJIIndoorLoc dataset with Wi-Fi RSSI features
2. **Exploratory Analysis**: RSSI distributions, AP coverage, spatial patterns
3. **Preprocessing**: RSSI cleaning, AP selection, feature engineering
4. **Modeling**: kNN-based building, floor, and position prediction
5. **Evaluation**: Comprehensive metrics and error analysis

### Key Findings:
- Building classification accuracy: ~XX%
- Floor classification accuracy: ~XX%
- Position error: ~XX meters (strict), ~XX meters (overall)
- Errors are significantly lower when building/floor are correct

### Next Steps:
- Try XGBoost or neural network models
- Implement hierarchical prediction
- Add device-specific calibration
- Deploy as FastAPI service